# Implementing the ResNet-34 Architecture from Scratch

In [ ]:
%%writefile ddp_train.py
"""
Standalone DDP training script for ResNet-34 on Food101.

This file exists (rather than living in the notebook) because torch.multiprocessing.spawn
uses the "spawn" start method: each child process re-imports __main__ to find the target
function.
"""

import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
from typing import Tuple, Dict, List
from tqdm.auto import tqdm


# Reproducibility 
def set_seed(seed: int = 42):
    """Set's seed of general PyTorch processes and cuda processes

    Args:
        seed (int, optional): random seed for the processes
    """

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)


# Model
class ResBlock(nn.Module):
    """Residual block (skip connection)."""
    expansion = 1

    def __init__(self, in_C, out_C, stride=1):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=in_C, out_channels=out_C, kernel_size=3,
                      stride=stride, padding=1, bias=False),        # bias = False cause they get cancelled out in BatchNorm
            nn.BatchNorm2d(out_C),
            nn.ReLU(inplace=True)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=out_C, out_channels=out_C, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_C)
        )
        self.relu = nn.ReLU()

        self.skip = nn.Sequential()
        if stride != 1 or in_C != out_C:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels=in_C, out_channels=out_C, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_C)
            )

    def forward(self, x):
        residual = self.skip(x)
        out = self.layer2(self.layer1(x))
        out += residual
        return self.relu(out)


class ResNet(nn.Module):
    """ResNet-34 architecture"""
    def __init__(self, in_channels: int, out_channels: int, blocks: list, block=ResBlock):
        super().__init__()
        self.in_channels = 64

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=64, kernel_size=7,
                      stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.block1 = self._make_layer(planes=64, blocks=blocks[0])
        self.block2 = self._make_layer(planes=128, blocks=blocks[1], stride=2)
        self.block3 = self._make_layer(planes=256, blocks=blocks[2], stride=2)
        self.block4 = self._make_layer(planes=512, blocks=blocks[3], stride=2)

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Linear(in_features=512 * block.expansion, out_features=out_channels)

    def _make_layer(self, planes: int, blocks: int, stride=1, block=ResBlock) -> nn.Sequential:
        """Generates a Sequential repeatative resblock connections
        Args:
            planes (int): Number of output channels for the block.
            blocks (int): Number of blocks to create.
            stride (int, optional): Stride for the first block. Defaults to 1.
            block (ResBlock, optional): Block type to use. Defaults to ResBlock.
        Returns:
            nn.Sequential: A sequential container of the created blocks.
        """

        layers = [block(self.in_channels, planes, stride=stride)]
        self.in_channels = planes * block.expansion     # to update the inchannels after every block consturction for the next block
        for _ in range(1, blocks):
            layers.append(block(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.gap(self.block4(self.block3(self.block2(self.block1(self.layer1(x))))))
        out = torch.flatten(out, start_dim=1)
        return self.fc(out)


# DDP setup
def setup_ddp(rank, world_size):
    """Setup for distributed data parallel (DDP) training.
    Args:
        rank (int): Rank of the current process.
        world_size (int): Total number of processes.
    """

    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'     # pick any free port

    # Initialize process gropup explicitly
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)


def cleanup_ddp():
    """Cleanup for distributed data parallel (DDP) training."""
    dist.destroy_process_group()


# Train / test steps 
def train_step(model, dataloader, loss_fn, optimizer, device) -> Tuple[float, float]:
    """Performs a single training step (epoch) for the model.
    Args:
        model (nn.Module): The model to train.
        dataloader (DataLoader): DataLoader for the training data.
        loss_fn: Loss function to use.
        optimizer: Optimizer to use.
        device: Device to run the training on.
    Returns:
        Tuple[float, float]: Average loss and accuracy for the epoch.
        example: (0.5, 0.8) -> 50% loss, 80% accuracy
    """

    model.train()
    train_loss, train_correct, train_samples = 0, 0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y)

        train_loss += loss.item() * len(y)
        train_correct += (y_pred.argmax(dim=1) == y).sum().item()
        train_samples += len(y)

        loss.backward()
        optimizer.step()

    metrics = torch.tensor([train_loss, train_correct, train_samples], device=device)
    dist.all_reduce(metrics, op=dist.ReduceOp.SUM)

    return metrics[0].item() / metrics[2].item(), metrics[1].item() / metrics[2].item()


def test_step(model, dataloader, loss_fn, device) -> Tuple[float, float]:
    """Performs a single testing step (epoch) for the model.
    Args:
        model (nn.Module): The model to test.
        dataloader (DataLoader): DataLoader for the testing data.
        loss_fn: Loss function to use.
        device: Device to run the testing on.
    Returns:
        Tuple[float, float]: Average loss and accuracy for the epoch.
        example: (0.5, 0.8) -> 50% loss, 80% accuracy
    """

    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            test_pred = model(X)
            loss = loss_fn(test_pred, y)

            total_loss += loss.item() * len(y)
            total_correct += (test_pred.argmax(dim=1) == y).sum().item()
            total_samples += len(y)

        metrics = torch.tensor([total_loss, total_correct, total_samples], device=device)
        dist.all_reduce(metrics, op=dist.ReduceOp.SUM)

    return metrics[0].item() / metrics[2].item(), metrics[1].item() / metrics[2].item()


# Dataloaders
def prepare_dataloaders(train_dataset, test_dataset, rank: int, world_size: int, batch_size: int = 32):
    """Prepares the dataloaders for distributed training.
    Args:
        train_dataset: Training dataset.
        test_dataset: Testing dataset.
        rank (int): Rank of the current process.
        world_size (int): Total number of processes.
        batch_size (int, optional): Batch size for the dataloaders. Defaults to 32.
    Returns:
        Tuple[DataLoader, DataLoader, DistributedSampler]: Training dataloader, testing dataloader, and training sampler.
    """
    
    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)
    test_sampler = DistributedSampler(test_dataset, num_replicas=world_size, rank=rank, shuffle=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler,
                              pin_memory=True, num_workers=2, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, sampler=test_sampler,
                             pin_memory=True, num_workers=2, shuffle=False)

    return train_loader, test_loader, train_sampler


# Train loop
def train(model, train_loader, test_loader, train_sampler, loss_fn, optimizer, scheduler,
         epochs, device, rank=0) -> Dict[str, List]:
    """Trains the model for a given number of epochs and evaluates on the test set.
    Args:
        model (nn.Module): The model to train.
        train_loader (DataLoader): DataLoader for the training data.
        test_loader (DataLoader): DataLoader for the testing data.
        train_sampler (DistributedSampler): Sampler for the training data.
        loss_fn: Loss function to use.
        optimizer: Optimizer to use.
        scheduler: Learning rate scheduler to use.
        epochs (int): Number of epochs to train for.
        device: Device to run the training on.
        rank (int, optional): Rank of the current process. Defaults to 0.
    Returns:
        Dict[str, List]: Dictionary containing training and testing loss and accuracy for each epoch.
        example: {"train_loss": [...], "train_acc": [...], "test_loss": [...], "test_acc": [...]}
    """
    
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    for epoch in tqdm(range(epochs), disable=(rank != 0)):
        train_sampler.set_epoch(epoch)

        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, device)
        test_loss, test_acc = test_step(model, test_loader, loss_fn, device)
        scheduler.step()

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if rank == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | "
                 f"Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    return results


# Worker
def whole_worker(rank: int, world_size: int, train_dataset, test_dataset):
    """Worker function for distributed training.
    Args:
        rank (int): Rank of the current process.
        world_size (int): Total number of processes.
        train_dataset: Training dataset.
        test_dataset: Testing dataset.
    """

    from torch.nn.parallel import DistributedDataParallel as DDP

    setup_ddp(rank, world_size)
    device = torch.device(f"cuda:{rank}")
    set_seed(42 + rank)

    train_loader, test_loader, train_sampler = prepare_dataloaders(
        train_dataset=train_dataset, test_dataset=test_dataset,
        rank=rank, world_size=world_size, batch_size=32
    )

    resnet = ResNet(in_channels=3, out_channels=101, blocks=[3, 4, 6, 3]).to(device)
    resnet = DDP(resnet, device_ids=[rank], output_device=rank)

    EPOCHS = 30
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4

    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    optimizer = torch.optim.Adam(resnet.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    results = train(
        model=resnet, train_loader=train_loader, test_loader=test_loader,
        train_sampler=train_sampler, loss_fn=loss_fn, optimizer=optimizer,
        scheduler=scheduler, epochs=EPOCHS, device=device, rank=rank
    )

    if rank == 0:
        # 1. Save your PyTorch outputs
        torch.save(resnet.module.state_dict(), "resnet34_food101.pth")
        torch.save(results, "training_results.pth")
        print("--> Saved best checkpoint and results successfully!")
    
        # 2. Define your archive name and the files to include
        archive_name = "food101_results.zip"
        files_to_zip = ["resnet34_food101.pth", "training_results.pth"]
    
        # 3. Create the targeted zip archive
        with zipfile.ZipFile(archive_name, "w", zipfile.ZIP_DEFLATED) as zipf:
            for file in files_to_zip:
                if os.path.exists(file):
                    zipf.write(file)
                    print(f"Added {file} to {archive_name}")
    
        # 4. Optional: Remove the raw .pth files if you ONLY want the single .zip in the output
        # for file in files_to_zip:
        #     os.remove(file)
    
        print(f"--> Custom archive created: {archive_name}")

    cleanup_ddp()


# Entry point
if __name__ == "__main__":
    """
    Entry point for the DDP training script.
    This script is designed to be run with torch.multiprocessing.spawn, which will
    create multiple processes for distributed training.
    """
    data_transform = transforms.Compose([
        transforms.Resize(size=(232, 232)),
        transforms.RandomCrop(size=(224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.TrivialAugmentWide(num_magnitude_bins=31),
        transforms.ToTensor()
    ])

    train_food = datasets.Food101(root='data', split='train', transform=data_transform,
                                  download=True, target_transform=None)
    test_food = datasets.Food101(root='data', split='test', transform=data_transform,
                                 download=True, target_transform=None)

    WORLD_SIZE = torch.cuda.device_count()
    print(f"Starting DDP across {WORLD_SIZE} GPUs...")

    mp.spawn(
        whole_worker,
        args=(WORLD_SIZE, train_food, test_food),
        nprocs=WORLD_SIZE,
        join=True
    )


In [ ]:
import subprocess

process = subprocess.Popen(
    ["python", "-u", "ddp_train.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
print(f"\nExit code: {process.returncode}")


In [ ]:
import matplotlib.pyplot as plt
import torch

# load results
results = torch.load("training_results.pth")
RAN = range(len(results['train_loss']))

results

In [ ]:
plt.plot(RAN, results['train_loss'], c = "r", label = "Train Loss")
plt.plot(RAN, results['test_loss'], c = "g", label = "Test Loss")
plt.title("Loss Curves")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(RAN, results['train_acc'], c = "r", label = "Train Acc")
plt.plot(RAN, results['test_acc'], c = "g", label = "Test Acc")
plt.title("Accuracy Curves")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# load the model
from ddp_train import ResNet,ResBlock

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiating ResNet
resnet = ResNet(in_channels=3, out_channels=101, blocks=[3, 4, 6, 3])

state_dict = torch.load("resnet34_food101.pth", weights_only=True)
resnet.load_state_dict(state_dict)

# 3. Move model to device
resnet = resnet.to(device)

# Going Modular

In [ ]:
# Imports

# import os

# # torch general imports
# import torch
# from torch import nn
# from torch.utils.data import DataLoader
# from torchvision import datasets, transforms

# # torch DDP imports
# import torch.distributed as dist
# import torch.multiprocessing as mp
# from torch.utils.data.distributed import DistributedSampler

# # Other
# from typing import Tuple, Dict, List
# from tqdm.auto import tqdm


## Data_setup Script

In [ ]:
# %%writefile data_setup.py

# """
# Food101 dataset/transform construction and distributed dataloaders 
# """

# from typing import Tuple

# from torch.utils.data import DataLoader, Dataset
# from torch.utils.data.distributed import DistributedSampler
# from torchvision import datasets, transforms

# NUM_WORKERS = 2


# def build_transforms() -> transforms.Compose:
#     return transforms.Compose([
#         transforms.Resize(size=(232, 232)),
#         transforms.RandomCrop(size=(224, 224)),
#         transforms.RandomHorizontalFlip(p=0.5),
#         transforms.TrivialAugmentWide(num_magnitude_bins=31),
#         transforms.ToTensor(),
#     ])


# def create_datasets(root: str = "data") -> Tuple[Dataset, Dataset]:
#     transform = build_transforms()

#     train_dataset = datasets.Food101(root=root, split="train", transform=transform, download=True)
#     test_dataset = datasets.Food101(root=root, split="test", transform=transform, download=True)

#     return train_dataset, test_dataset


# def create_dataloaders(
#     train_dataset: Dataset,
#     test_dataset: Dataset,
#     rank: int,
#     world_size: int,
#     batch_size: int = 32,
# ) -> Tuple[DataLoader, DataLoader, DistributedSampler]:
#     train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)
#     test_sampler = DistributedSampler(test_dataset, num_replicas=world_size, rank=rank, shuffle=False)

#     train_loader = DataLoader(
#         train_dataset, batch_size=batch_size, sampler=train_sampler,
#         pin_memory=True, num_workers=NUM_WORKERS,
#     )
#     test_loader = DataLoader(
#         test_dataset, batch_size=batch_size, sampler=test_sampler,
#         pin_memory=True, num_workers=NUM_WORKERS,
#     )

#     return train_loader, test_loader, train_sampler


## Model building

In [ ]:
# %%writefile model_builder.py
# """
# ResNet-34 model builder script

# This script contains the classes to instantiate the resnet34 
# model architecture
# """

# import torch
# from torch import nn

# class ResBlock(nn.Module):
#     """Instantiatig ResBlock"""

#     expansion = 1

#     def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
#         super().__init__()

#         self.layer1 = nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#         )
#         self.layer2 = nn.Sequential(
#             nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#         )
#         self.relu = nn.ReLU()

#         self.skip = nn.Sequential()
#         if stride != 1 or in_channels != out_channels:
#             self.skip = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
#                 nn.BatchNorm2d(out_channels),
#             )

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         residual = self.skip(x)
#         out = self.layer2(self.layer1(x))
#         out += residual
#         return self.relu(out)

# class ResNet(nn.Module):
#     """Instantiaing ResNet34 build"""
#     def __init__(self, in_channels: int, out_channels: int, blocks: list, block=ResBlock):
#         super().__init__()
#         self.in_channels = 64

#         self.layer1 = nn.Sequential(
#             nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
#             nn.BatchNorm2d(64),
#             nn.ReLU(inplace=True),
#             nn.MaxPool2d(kernel_size=3, stride=2),
#         )

#         self.block1 = self._make_layer(block, 64, blocks[0])
#         self.block2 = self._make_layer(block, 128, blocks[1], stride=2)
#         self.block3 = self._make_layer(block, 256, blocks[2], stride=2)
#         self.block4 = self._make_layer(block, 512, blocks[3], stride=2)

#         self.gap = nn.AdaptiveAvgPool2d((1, 1))
#         self.fc = nn.Linear(512 * block.expansion, out_channels)

#     def _make_layer(self, planes: int, blocks: int, stride = 1, block=ResBlock)->nn.Sequential:
#         """Generates a Sequential repeatative resblock connections"""
#         layers = [block(self.in_channels, planes, stride = stride)]
#         self.in_channels = planes * block.expansion
#         for _ in range(1, blocks):
#             layers.append(block(planes, planes))
#         return nn.Sequential(*layers)

#     def forward(self,x):
#         out = self.gap(self.block4(self.block3(self.block2(self.block1(self.layer1(x))))))
#         out = torch.flatten(out, start_dim=1)
#         return self.fc(out)

## DDP setup

In [ ]:
# %%writefile distributed_utils.py

# """
# This script contrains the functions required for PyTorch DDP
# setup and cleanup
# """

# import os

# import torch
# import torch.distributed as dist


# def set_seed(seed: int = 42) -> None:
#     """Sets the seed for CPU and CUDA RNGs."""
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed(seed)


# def setup_ddp(rank: int, world_size: int) -> None:
#     """Initializes the process group for single-node multi-GPU training."""
#     os.environ["MASTER_ADDR"] = "localhost"
#     os.environ["MASTER_PORT"] = "12355"

#     dist.init_process_group("nccl", rank=rank, world_size=world_size)
#     torch.cuda.set_device(rank)


# def cleanup_ddp() -> None:
#     """Tears down the process group."""
#     dist.destroy_process_group()

## Training and Testing engine

In [ ]:
# %%writefile engine.py
# """
# Contains the training and testing loop of a pytorch model on 
# DDP setup
# """

# from typing import Dict, List, Tuple

# import torch
# import torch.distributed as dist
# from torch.utils.data import DataLoader
# from torch.utils.data.distributed import DistributedSampler
# from tqdm.auto import tqdm


# def train_step(
#     model: torch.nn.Module,
#     dataloader: DataLoader,
#     loss_fn: torch.nn.Module,
#     optimizer: torch.optim.Optimizer,
#     device: torch.device,
# ) -> Tuple[float, float]:
#     model.train()
#     train_loss, train_correct, train_samples = 0.0, 0, 0

#     for X, y in dataloader:
#         X, y = X.to(device), y.to(device)
#         optimizer.zero_grad()

#         y_pred = model(X)
#         loss = loss_fn(y_pred, y)

#         train_loss += loss.item() * len(y)
#         train_correct += (y_pred.argmax(dim=1) == y).sum().item()
#         train_samples += len(y)

#         loss.backward()
#         optimizer.step()

#     metrics = torch.tensor([train_loss, train_correct, train_samples], device=device)
#     dist.all_reduce(metrics, op=dist.ReduceOp.SUM)

#     return metrics[0].item() / metrics[2].item(), metrics[1].item() / metrics[2].item()


# def test_step(
#     model: torch.nn.Module,
#     dataloader: DataLoader,
#     loss_fn: torch.nn.Module,
#     device: torch.device,
# ) -> Tuple[float, float]:
#     model.eval()
#     total_loss, total_correct, total_samples = 0.0, 0, 0

#     with torch.inference_mode():
#         for X, y in dataloader:
#             X, y = X.to(device), y.to(device)
#             test_pred = model(X)
#             loss = loss_fn(test_pred, y)

#             total_loss += loss.item() * len(y)
#             total_correct += (test_pred.argmax(dim=1) == y).sum().item()
#             total_samples += len(y)

#         metrics = torch.tensor([total_loss, total_correct, total_samples], device=device)
#         dist.all_reduce(metrics, op=dist.ReduceOp.SUM)

#     return metrics[0].item() / metrics[2].item(), metrics[1].item() / metrics[2].item()


# def train(
#     model: torch.nn.Module,
#     train_loader: DataLoader,
#     test_loader: DataLoader,
#     train_sampler: DistributedSampler,
#     loss_fn: torch.nn.Module,
#     optimizer: torch.optim.Optimizer,
#     scheduler: torch.optim.lr_scheduler.LRScheduler,
#     epochs: int,
#     device: torch.device,
#     rank: int = 0,
# ) -> Dict[str, List[float]]:
#     results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

#     for epoch in tqdm(range(epochs), disable=(rank != 0)):
#         train_sampler.set_epoch(epoch)

#         train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, device)
#         test_loss, test_acc = test_step(model, test_loader, loss_fn, device)
#         scheduler.step()

#         results["train_loss"].append(train_loss)
#         results["train_acc"].append(train_acc)
#         results["test_loss"].append(test_loss)
#         results["test_acc"].append(test_acc)

#         if rank == 0:
#             print(
#                 f"Epoch {epoch + 1:02d}/{epochs} | Train Loss: {train_loss:.4f} | "
#                 f"Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
#             )

#     return results


## Main training script

In [ ]:
# %%writefile train.py
# """
# Contains train function that puts everything tog
# """

# import argparse

# import torch
# import torch.multiprocessing as mp
# from torch.nn.parallel import DistributedDataParallel as DDP

# import data_setup
# import distributed_utils
# import engine
# import model_builder


# def parse_args() -> argparse.Namespace:
#     parser = argparse.ArgumentParser(description="Train ResNet-34 on Food101 with DDP")
#     parser.add_argument("--epochs", type=int, default=15)
#     parser.add_argument("--learning-rate", type=float, default=0.001)
#     parser.add_argument("--weight-decay", type=float, default=1e-4)
#     parser.add_argument("--batch-size", type=int, default=32)
#     parser.add_argument("--label-smoothing", type=float, default=0.1)
#     parser.add_argument("--eta-min", type=float, default=1e-6, help="Minimum LR for the cosine scheduler")
#     parser.add_argument("--seed", type=int, default=42)
#     parser.add_argument("--data-root", type=str, default="data")
#     parser.add_argument("--checkpoint-path", type=str, default="resnet34_food101.pth")
#     parser.add_argument("--results-path", type=str, default="training_results.pth")
#     return parser.parse_args()


# def worker(rank: int, world_size: int, train_dataset, test_dataset, args: argparse.Namespace) -> None:
#     distributed_utils.setup_ddp(rank, world_size)
#     device = torch.device(f"cuda:{rank}")
#     distributed_utils.set_seed(args.seed + rank)

#     train_loader, test_loader, train_sampler = data_setup.create_dataloaders(
#         train_dataset, test_dataset, rank=rank, world_size=world_size, batch_size=args.batch_size
#     )

#     model = model_builder.ResNet(in_channels=3, out_channels=101, blocks=[3, 4, 6, 3]).to(device)
#     model = DDP(model, device_ids=[rank], output_device=rank)

#     loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=args.label_smoothing).to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=args.eta_min)

#     results = engine.train(
#         model=model,
#         train_loader=train_loader,
#         test_loader=test_loader,
#         train_sampler=train_sampler,
#         loss_fn=loss_fn,
#         optimizer=optimizer,
#         scheduler=scheduler,
#         epochs=args.epochs,
#         device=device,
#         rank=rank,
#     )

#     if rank == 0:
#         torch.save(model.module.state_dict(), args.checkpoint_path)
#         torch.save(results, args.results_path)
#         print("--> Saved checkpoint and results successfully!")

#     distributed_utils.cleanup_ddp()


# if __name__ == "__main__":
#     args = parse_args()

#     train_dataset, test_dataset = data_setup.create_datasets(root=args.data_root)
#     world_size = torch.cuda.device_count()

#     print(f"Starting DDP across {world_size} GPUs...")
#     mp.spawn(worker, args=(world_size, train_dataset, test_dataset, args), nprocs=world_size, join=True)


## Prediction script

In [ ]:
# %%writefile predict.py
# """
# Makes prediction on random data instances using a PyTorch Model
# """
# import argparse

# import matplotlib.pyplot as plt
# import torch

# import data_setup
# import model_builder
# import utils


# def parse_args() -> argparse.Namespace:
#     parser = argparse.ArgumentParser(description="Visualize ResNet-34 predictions on Food101 test samples")
#     parser.add_argument("--checkpoint-path", type=str, default="resnet34_food101.pth")
#     parser.add_argument("--data-root", type=str, default="data")
#     parser.add_argument("--rows", type=int, default=4)
#     parser.add_argument("--cols", type=int, default=4)
#     return parser.parse_args()


# def visualize_predictions(
#     model: torch.nn.Module,
#     dataset,
#     class_names: list,
#     device: torch.device,
#     rows: int = 4,
#     cols: int = 4,
# ) -> None:
#     model.eval()
#     fig = plt.figure(figsize=(15, 15))

#     for i in range(1, rows * cols + 1):
#         idx = torch.randint(1, len(dataset), size=[1]).item()
#         img, label = dataset[idx]

#         with torch.inference_mode():
#             pred = model(img.to(device).unsqueeze(dim=0)).argmax(dim=1)

#         plt.subplot(rows, cols, i)
#         plt.imshow(img.permute(1, 2, 0))
#         plt.title(f"Original: {class_names[label]}\nPredicted: {class_names[pred]}")
#         plt.axis(False)

#     plt.show()


# if __name__ == "__main__":
#     args = parse_args()
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     _, test_dataset = data_setup.create_datasets(root=args.data_root)
#     class_names = test_dataset.classes

#     model = model_builder.ResNet(in_channels=3, out_channels=101, blocks=[3, 4, 6, 3])
#     model = utils.load_model(model, args.checkpoint_path, device)

#     visualize_predictions(model, test_dataset, class_names, device, rows=args.rows, cols=args.cols)


## Other utils

In [ ]:
# %%writefile utils.py

# """
# Some helper functions
# """
# from pathlib import Path
# from typing import Dict, List

# import matplotlib.pyplot as plt
# import torch


# def save_model(model: torch.nn.Module, target_dir: str, model_name: str) -> None:
#     assert model_name.endswith(".pth") or model_name.endswith(".pt"), \
#         "model_name should end with '.pt' or '.pth'"

#     target_dir_path = Path(target_dir)
#     target_dir_path.mkdir(parents=True, exist_ok=True)
#     torch.save(model.state_dict(), target_dir_path / model_name)


# def load_model(model: torch.nn.Module, checkpoint_path: str, device: torch.device) -> torch.nn.Module:
#     state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
#     model.load_state_dict(state_dict)
#     return model.to(device)


# def plot_curves(results: Dict[str, List[float]]) -> None:
#     epochs = range(len(results["train_loss"]))

#     plt.figure()
#     plt.plot(epochs, results["train_loss"], c="r", label="Train Loss")
#     plt.plot(epochs, results["test_loss"], c="g", label="Test Loss")
#     plt.title("Loss Curves")
#     plt.grid(True)
#     plt.legend()
#     plt.show()

#     plt.figure()
#     plt.plot(epochs, results["train_acc"], c="r", label="Train Acc")
#     plt.plot(epochs, results["test_acc"], c="g", label="Test Acc")
#     plt.title("Accuracy Curves")
#     plt.grid(True)
#     plt.legend()
#     plt.show()